<a href="https://colab.research.google.com/github/trbpook/ADA-Lab2/blob/main/sc2001_project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

(a) Suppose the input graph G = (V, E) is stored in an adjacency matrix and we
use an array for the priority queue. Implement the Dijkstra’s algorithm using this
setting and analyze its time complexity with respect to |V| and |E| both
theoretically and empirically

In [ ]:
import math

def dijkstra_matrix(graph, source):
  V = len(graph)

  dist = [math.inf] * V # initialise all dist to infinity
  visited = [False] * V # avoid visiting repeatedly

  dist[source] = 0

  for _ in range(V):

    # find the unvisited vertex with the smallest current distance

    u = -1
    min_dist = math.inf

    for v in range(V):
      if not visited[v] and dist[v] < min_dist:
        u = v
        min_dist = dist[v]

    # no reachable vertex left
    if u == -1:
      break

    visited[u] = True


    # relax all possible neighbours
    for v in range(V):
      weight = graph[u][v]

      if (
          not visited[v]
          and weight != 0
          and dist [u] + weight < dist[v]
      ):
        dist[v] = dist[u] + weight

  return dist




Testing

In [ ]:
graph = [
    [0, 4, 1, 0],
    [4, 0, 2, 1],
    [1, 2, 0, 5],
    [0, 1, 5, 0]
]

distances = dijkstra_matrix(graph, 0)

print(distances)

[0, 3, 1, 4]


### Theoretical Analysis

Firstly, finding the unvisited vertex with the smallest current distance requires scanning the array, which takes (O(|V|)) time.  

Secondly, relaxing all possible neighbours requires scanning one row of the adjacency matrix, which also takes (O(|V|)) time.  

These two steps are repeated up to (|V|) times. Therefore, the overall time complexity is:
 O(|V|x|V|) =  O(|V|²).

Emperical data

In [ ]:
import random
import time

def generate_graph(V, density):
    graph = [[0] * V for _ in range(V)]

    for i in range(V):
        for j in range(i + 1, V):
            if random.random() < density:
                weight = random.randint(1, 20)
                graph[i][j] = weight
                graph[j][i] = weight

    return graph


def measure_runtime(graph, runs=5): # run 5 times and take the average
    total = 0

    for _ in range(runs):
        start = time.perf_counter()
        dijkstra_matrix(graph, 0)
        end = time.perf_counter()

        total += end - start

    return total / runs

print("Experiment 1: |V| varies, fixed |E|.")
print("")

V_values = [100, 200, 400, 800, 1200]

results = []

for V in V_values:
    graph = generate_graph(V, 0.2)
    runtime = measure_runtime(graph)

    results.append((V, runtime))

    print(f"V = {V}, runtime = {runtime:.6f}s")

print("Experiment 2: |V| varies, fixed |E|.")
print("")

V = 500
densities = [0.1, 0.2, 0.4, 0.6, 0.8]

for density in densities:
    graph = generate_graph(V, density)

    runtime = measure_runtime(graph)

    E = sum(
        1
        for i in range(V)
        for j in range(i + 1, V)
        if graph[i][j] != 0
    )

    print(f"E = {E}, runtime = {runtime:.6f}s")


Experiment 1: |V| varies, fixed |E|.

V = 100, runtime = 0.001656s
V = 200, runtime = 0.006391s
V = 400, runtime = 0.024876s
V = 800, runtime = 0.104958s
V = 1200, runtime = 0.233289s
Experiment 2: |V| varies, fixed |E|.

E = 12577, runtime = 0.038099s
E = 24844, runtime = 0.041165s
E = 49915, runtime = 0.044278s
E = 74777, runtime = 0.043950s
E = 99732, runtime = 0.043884s


### Empirical Analysis

The runtime increases significantly as (|V|) increases and follows an approximately quadratic trend, which is consistent with the theoretical complexity of (O(|V|^2)). When (|V|) is fixed and (|E|) is varied, the runtime changes relatively little because the adjacency matrix implementation scans all (|V|) possible neighbours regardless of the actual number of edges.

### (b). Using adjacency list and min. heap for priority queue

In [ ]:
import heapq

def dijkstra_heap(adj_list, source):
    n = len(adj_list)
    distances = [float('inf')] * n
    visited = [False] * n

    # 1. Initialize distance and heap
    distances[source] = 0
    pq = [(0, source)]  # stores (distance, vertex)

    # 2. Main loop
    while pq:
        # Instantly pops the node with the minimum distance
        current_dist, u = heapq.heappop(pq)

        # Skip if already finalized
        if visited[u]:
            continue
        visited[u] = True

        # 3. Explore neighbors from adjacency list
        for v, weight in adj_list[u]:
            if not visited[v]:
                new_dist = current_dist + weight
                if new_dist < distances[v]:
                    distances[v] = new_dist
                    # Add new shorter distance to the heap
                    heapq.heappush(pq, (new_dist, v))

    return distances

#### Theoretical analysis
By replacing the linear matrix scan (O(|V|) per node) with a min-heap (O(log|V|)) and an adjacency list (O(|E|) total edge traversals), the overall complexity improves from O(|V|²) to O((|V| + |E|)log |V|).

#### Empirical analysis

In [ ]:
import time
import random
import matplotlib.pyplot as plt

from dijkstra_b import dijkstra_heap

def generate_random_adj_list(num_v, num_e):
    adj_list = [[] for _ in range(num_v)]
    edges_added = set()

    # Pre-populate edges to ensure connectivity (spanning tree structure)
    for i in range(1, num_v):
        parent = random.randint(0, i - 1)
        weight = random.randint(1, 100)
        adj_list[parent].append((i, weight))
        edges_added.add((parent, i))

    # Add the remaining edges randomly
    while len(edges_added) < num_e:
        u = random.randint(0, num_v - 1)
        v = random.randint(0, num_v - 1)
        if u != v and (u, v) not in edges_added:
            weight = random.randint(1, 100)
            adj_list[u].append((v, weight))
            edges_added.add((u, v))

    return adj_list


def benchmark_part_b(trials=3):
    print("--- Running Test 1: Varying |V| (Fixed Degree, |E| = 4 * |V|) ---")
    v_sizes = [1000, 2500, 5000, 10000, 20000, 40000]
    v_times = []

    for v in v_sizes:
        e = 4 * v
        trial_runtimes = []
        for _ in range(trials):
            graph = generate_random_adj_list(v, e)
            start = time.perf_counter()
            dijkstra_heap(graph, source=0)
            trial_runtimes.append(time.perf_counter() - start)

        avg_time = sum(trial_runtimes) / trials
        v_times.append(avg_time)
        print(f"|V| = {v:5d} | |E| = {e:6d} | Avg Time: {avg_time:.5f}s")

    print("\n--- Running Test 2: Varying |E| (Fixed |V| = 2000) ---")
    fixed_v = 2000
    # From sparse (~2k edges) up to dense (~200k edges)
    e_sizes = [2000, 10000, 25000, 50000, 100000, 200000]
    e_times = []

    for e in e_sizes:
        trial_runtimes = []
        for _ in range(trials):
            graph = generate_random_adj_list(fixed_v, e)
            start = time.perf_counter()
            dijkstra_heap(graph, source=0)
            trial_runtimes.append(time.perf_counter() - start)

        avg_time = sum(trial_runtimes) / trials
        e_times.append(avg_time)
        print(f"|V| = {fixed_v:5d} | |E| = {e:6d} | Avg Time: {avg_time:.5f}s")

    # Plotting the results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Varying |V|
    ax1.plot(v_sizes, v_times, marker='o', color='royalblue', label='Empirical Runtime')
    ax1.set_title("Part (b): Runtime vs |V| (Sparse: |E| = 4|V|)")
    ax1.set_xlabel("Number of Vertices (|V|)")
    ax1.set_ylabel("Execution Time (s)")
    ax1.grid(True)
    ax1.legend()

    # Plot 2: Varying |E|
    ax2.plot(e_sizes, e_times, marker='s', color='darkorange', label='Empirical Runtime')
    ax2.set_title("Part (b): Runtime vs |E| (Fixed |V| = 2000)")
    ax2.set_xlabel("Number of Edges (|E|)")
    ax2.set_ylabel("Execution Time (s)")
    ax2.grid(True)
    ax2.legend()

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    benchmark_part_b()